In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import networkx as nx
import numpy as np
from rich import print as rich_print

from helpers import classify_cage_search_result
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
    fock_space_automorphism_diagnostic_for_cage_state,
)
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.models.couplings import DirectedPlaquetteCoupling
from qlinks.visualizer import (
    LinkVisualStyle,
    BasisGridVisualizer,
    LocalBasisGridVisualizer,
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
)

## Model definition

Here goes the definition of the model, with optional plaquette-dependent coupling, which can be either a `Dict` or `Callable`

In [ ]:
def coup_kin(plaquette_id: int) -> DirectedPlaquetteCoupling:
    if plaquette_id in [1, 9]:
        phase = 0.5 * np.pi
    elif plaquette_id in [7, 15]:
        phase = -0.5 * np.pi
    else:
        phase = 0
    return DirectedPlaquetteCoupling(
        forward=-np.exp(1j * phase),
    )

In [ ]:
model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    # charge_magnitude=1,
    coup_kin=-1.0,
    coup_pot=0.7,
)
print(model.allowed_sector_labels())
print(model.nonempty_sector_labels())
# print(model.lattice.plaquette_id_from_anchor((3, 3), kind="square"))

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="bitmask",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("H shape =", hamiltonian_matrix.shape)
print("H nnz =", hamiltonian_matrix.nnz)
print("K nnz =", kinetic_matrix.nnz)
print("V nnz =", potential_matrix.nnz)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

## Search for caged states

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="type1",
        # type1_kappas=(0,),
        # type2_kappas=(-2, 2),
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
        store_full_states=True,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = -1
record = search_result[signature, record_index]

state_vector = record.full_state
if state_vector is None:
    state_vector = search_result.full_state_matrix()[record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected caged state

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        collective_cancellation_mode="all_problematic_sum",
    ),
)

rich_print(report)
# rich_print(report.to_text(verbose=True))

## Visualize the caged state on Hamiltonian graph

In [ ]:
graph_visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    kinetic_matrix,
    include_self_loops=False,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=26,
        vertex_label_size=4,
    ),
)

graph_visualizer.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    edge_color_by="weight_phase",
    state_vector=state_vector,
    layout="kk",
    title=f"Fock-space graph colored by cage-state signed amplitude",
)

In [ ]:
small_viz = HamiltonianGraphVisualizer.cage_subgraph_from_sparse_matrix(
    build_result.kinetic,
    state_vector,
    classification_report=report,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=40,
        vertex_label_size=6,
    ),
)

small_viz.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    edge_color_by="weight_phase",
    layout="kk",
    title=f"The relevant subgraph colored by cage-state signed amplitude",
)

### Save the graph (optional)

In [ ]:
graph_visualizer.save_graph(
    f"../data/fock_graph.gexf",
    layout_backend="igraph",
    layout="kk",
    color_by="state_amplitude_real",
    state_vector=state_vector,
)

## Visualize the basis states for
* caged state
* interference zeros
* all basis states (optional)

In [ ]:
grid_visualizer = BasisGridVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    periodic_image_mode="positive_patch",
    collapse_duplicate_visual_links=False,
    site_label_style="sublattice_cell",
    # coordinate_transform=np.array(
    #     [
    #         [1.0, 0.0],
    #         [0.0, 0.72],
    #     ],
    #     dtype=float,
    # ),
    style=LinkVisualStyle(
        node_size=50.0,
        site_label_fontsize=4,
        link_label_fontsize=8,
        plaquette_symbol_fontsize=8,
        vulnerable_link_arrow_length_fraction=1.1,
    ),
)

In [ ]:
fig, ax = grid_visualizer.plot_cage_support(
    search_result,
    ncols=4,
    basis_configs=basis_configs,
    signature=signature,
    record_index=record_index,
    show_config_label=False,
    # figsize=(16, 8),
    # single_plot_kwargs={
    #     "with_site_labels": True,
    #     "with_site_values": False,
    #     "with_link_values": False,
    #     "with_link_ids": True,
    # },
)

In [ ]:
grid_visualizer.plot_interference_zeros(
    report,
    ncols=4,
    basis_configs=basis_configs,
    mechanism="all",
    show_config_label=False,
)

In [ ]:
grid_visualizer.plot(
    basis_configs,
    ncols=4,
    show_config_label=False,
    suptitle="All basis configurations",
)

In [ ]:
fig.savefig("honeycomb_qdm_6x4_w0-2_signature_04_record1.png", dpi=240, bbox_inches="tight")

## Local structure analysis

In [ ]:
structure = report.local_structure_report(
    basis_configs=basis_configs,
    state=record.full_state,
    model=model,
    decomposition="exact_support",
)

print(structure.to_text())

In [ ]:
local_visualizer = LocalBasisGridVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    # coordinate_transform=np.array(
    #     [
    #         [1.0, 0.0],
    #         [0.0, 0.72],
    #     ],
    #     dtype=float,
    # ),
    style=LinkVisualStyle(
        node_size=50.0,
        site_label_fontsize=4,
        link_label_fontsize=8,
        plaquette_symbol_fontsize=8,
        vulnerable_link_arrow_length_fraction=1.1,
    ),
)

In [ ]:
fig, axes = local_visualizer.plot_structure_report(
    structure,
    # reference_config=certified.basis.states[0],   # optional
    max_readouts=12,
    max_structures_per_readout=3,
    max_basis_states=4,
    include_frozen=False,
    max_frozen_per_readout=4,
    mode="auto",
    single_plot_kwargs={"with_site_labels": False},
)

In [ ]:
fig, axes = local_visualizer.plot_structure_readout(
    structure.readout_reports[3],
    # reference_config=certified.basis.states[0],   # optional
    max_structures=4,
    max_basis_states=4,
    include_frozen=False,
    max_frozen=6,
    mode="auto",
    coherent_plaquette_symbols="auto",
    frozen_plaquette_symbols="none",
    single_plot_kwargs={"with_site_labels": False},
)

In [ ]:
readouts = report.reduced_iz_local_rdm_readouts(
    basis_configs=basis_configs,
    state=state_vector,
    decomposition="exact_support",
    max_matrix_unit_terms=None,
)

print("number of terms:", len(readouts))
for readout in readouts:
    print("="*80)
    print("variable_indices:", readout.variable_indices)
    print("support_rank:", readout.support_rank)
    print("nullity:", readout.nullity)
    print("matrix units:")
    for term in readout.matrix_unit_terms:
        tag = "diag" if term.target_pattern == term.source_pattern else "off_diag"
        print(" ", term.coefficient, "|", term.target_pattern, "><", term.source_pattern, "|", tag)

## Classify all cages

In [ ]:
df, classification_reports = classify_cage_search_result(
    search_result,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=None,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
        collective_cancellation_mode="all_problematic_nullspace",
    ),
)

df

## Exploratory analysis

In [ ]:
auto = fock_space_automorphism_diagnostic_for_cage_state(
    kinetic_matrix=kinetic_matrix,
    cage_state=record.cage_state,
    tolerance=1e-10,
    weight_tolerance=1e-12,
    max_graph_vertices=128,
    max_automorphisms=256,
)

auto.to_summary_dict()

In [ ]:
v = state_vector

exp_kin = v.T @ kinetic_matrix @ v
exp_pot = v.T @ potential_matrix @ v
exp_H = v.T @ hamiltonian_matrix @ v
print(exp_kin, exp_pot, exp_H)
print(np.allclose(kinetic_matrix @ v, 0, atol=1e-10), np.linalg.norm(kinetic_matrix @ v))